# Benchmark audité du corpus GeNIS 2025 — pipeline expérimental de l'Article 1

**Papier cible :** *A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus* (Bahri, Jemili, Mosbah).

---

## Organisation du notebook

| § | Étape | Contenu |
|---|-------|---------|
| 1 | **Configuration** | environnement, constantes, dossiers, reprise automatique |
| 2 | **Exploration des données (EDA)** | les 4 intervalles, schéma, distribution des classes, chronologie des attaques |
| 3 | **Prétraitement** | unification des labels (9 classes), catégorisation des features, splits gelés, sondes de raccourci |
| 4 | **Modèles** | définition des 9 familles + autoencodeur (provenance BAg-IDS pour RNN/CNN/DNN) |
| 5 | **Entraînement** | conditions `full` → `clean` → **audit** → `audited` ; split temporel |
| 6 | **Évaluation** | tables, figures 300 dpi, calibration, coût, tests statistiques |
| 7 | **Export** | JSON consolidé + archive figures/tables pour la rédaction |

## Points de protocole (gelés)

- **Protocole propre** : scaler ajusté sur le train uniquement, distribution naturelle conservée, aucune rééquilibration du test.
- **Splits** : stratifié 60/20/20 × 5 graines **+ split temporel par classe** (pour chaque classe, les flux de test sont postérieurs à ceux d'entraînement). Le split chronologique *global* est dégénéré sur GeNIS (le test final ne contient que des classes jamais vues) — il est documenté comme **résultat** (§3.4) mais pas utilisé comme protocole.
- **Trois conditions de features** : `full` (avec colonnes positionnelles) → `clean` (sans) → `audited` (après liste noire issue de l'audit §5.2).

## Exécution

- Runtime **GPU** (T4 suffisant). Prérequis : `2-flows.zip` dans `MyDrive/GeNIS/`.
- *Exécution → Tout exécuter*. Chaque run est sauvegardé dans `MyDrive/GeNIS/article1_master/` :
  en cas de déconnexion, relancer — le notebook **reprend automatiquement** (les runs déjà faits sont sautés).
- Les résultats des exécutions précédentes (conditions `full`/`clean` stratifiées) **sont réutilisés tels quels**.

**À renvoyer à la fin : `article1_results.json` + `article1_figures_tables.zip`.**


## 1. Configuration

Environnement, constantes du protocole et reprise. Les avertissements non informatifs
(noms de features sklearn/LightGBM) sont désactivés pour garder une sortie lisible.


In [ ]:
# 1.1 — Environnement et constantes
import os, sys, glob, json, time, math, shutil, hashlib, pathlib, platform, itertools, warnings
import numpy as np, pandas as pd, sklearn, scipy
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display
try:
    import xgboost, lightgbm
except ImportError:
    %pip -q install xgboost lightgbm
    import xgboost, lightgbm

warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

# ---- constantes du protocole (gelees) ----
SEEDS        = [1, 2, 3, 4, 5]   # graines des splits stratifies
SPLIT_FRAC   = (0.60, 0.20, 0.20)
WINDOW_GAP_S = 1800              # 30 min sans flux = nouvelle fenetre d'activite
RUN_INTERVALS = True             # §6.4 : etude 5/10/30 s (~1-1.5 h)
RUN_COST      = True             # §6.5 : banc de cout CPU (~15 min)

GPU = bool(tf.config.list_physical_devices("GPU"))
print(f"python {platform.python_version()} | tf {tf.__version__} | sklearn {sklearn.__version__} "
      f"| xgb {xgboost.__version__} | lgbm {lightgbm.__version__}")
print("GPU :", "oui" if GPU else "NON (sections profondes lentes)")


In [ ]:
# 1.2 — Donnees et reprise automatique
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SAVE  = pathlib.Path("/content/drive/MyDrive/GeNIS/article1_master")
PROBS = SAVE / "probs"; FIGS = SAVE / "figures"; TABS = SAVE / "tables"
for p in (SAVE, PROBS, FIGS, TABS): p.mkdir(parents=True, exist_ok=True)

RES_PATH = SAVE / "article1_results.json"
RESULTS = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
RESULTS.setdefault("models", {}); RESULTS.setdefault("meta", {})
# purge : les runs de l'ancien split chronologique global (protocole degenere, remplace
# par le split temporel par classe) et de l'ancien audit sont invalides
RESULTS["models"] = {k: v for k, v in RESULTS["models"].items() if not k.endswith("|chrono")}
if RESULTS.get("audit", {}).get("version") != "v2-temporal":
    RESULTS.pop("audit", None)
    RESULTS["models"] = {k: v for k, v in RESULTS["models"].items() if "|audited|" not in k}
RESULTS["meta"].update({"updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                        "env": {"tf": tf.__version__, "sklearn": sklearn.__version__,
                                "xgb": xgboost.__version__, "lgbm": lightgbm.__version__, "gpu": GPU}})
def save_results():
    RES_PATH.write_text(json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")
save_results()
print("runs deja calcules (reutilises) :", len(RESULTS["models"]))

drive_zip = "/content/drive/MyDrive/GeNIS/2-flows.zip"
if not pathlib.Path("flows").exists():
    if pathlib.Path(drive_zip).exists():
        print("copie depuis Drive…"); shutil.copy(drive_zip, "2-flows.zip")
    else:
        print("telechargement Zenodo (~380 Mo)…")
        !wget -q --show-progress "https://zenodo.org/records/14919237/files/2-flows.zip?download=1" -O 2-flows.zip
    !unzip -o -q 2-flows.zip -d flows
csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))
assert csvs, "aucun CSV trouve"
print(len(csvs), "fichiers CSV")


## 2. Exploration des données (EDA)

GeNIS fournit le **même trafic** agrégé à quatre intervalles (5/10/30/60 s) — quatre vues,
jamais à mélanger. L'EDA établit : (i) la taille et le déséquilibre de chaque vue,
(ii) le schéma des features (exporteur HERA/Argus), (iii) la structure temporelle de la
capture — qui conditionne toute la suite.


In [ ]:
# 2.1 — Les quatre intervalles : volumes et part de trafic benin (Table 1 du papier)
INTERVALS = ["5", "10", "30", "60"]
if "interval_stats" not in RESULTS:
    st = {}
    for iv in INTERVALS:
        counts = {pathlib.Path(f).stem: sum(1 for _ in open(f, "rb")) - 1
                  for f in csvs if f"flows-{iv}-sec" in f}
        tot = sum(counts.values()); ben = sum(v for k, v in counts.items() if k.startswith("benign"))
        st[iv] = {"files": counts, "total": tot, "benign": ben, "benign_share": ben / tot}
    RESULTS["interval_stats"] = st; save_results()

t1 = pd.DataFrame({iv: {"flux total": RESULTS["interval_stats"][iv]["total"],
                        "flux benins": RESULTS["interval_stats"][iv]["benign"],
                        "part benin (%)": round(RESULTS["interval_stats"][iv]["benign_share"]*100, 2)}
                   for iv in INTERVALS}).T
t1.index.name = "intervalle (s)"
display(t1)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(INTERVALS, [RESULTS["interval_stats"][iv]["total"] for iv in INTERVALS], color="steelblue")
ax[0].set_title("Flux par intervalle"); ax[0].set_xlabel("intervalle (s)")
ax[1].bar(INTERVALS, [RESULTS["interval_stats"][iv]["benign_share"]*100 for iv in INTERVALS], color="seagreen")
ax[1].set_title("Part de trafic benin (%)"); ax[1].set_xlabel("intervalle (s)")
plt.tight_layout(); plt.savefig(FIGS / "fig_eda_intervals.png"); plt.show()
# Lecture : la part de benin varie avec l'intervalle — l'accuracy brute n'est comparable
# ni entre intervalles ni avec d'autres corpus. D'ou macro-F1 / MCC dans tout le papier.


In [ ]:
# 2.2 — Tranche 60 s : chargement, unification des labels (11 -> 9 classes), schema
def load_slice(iv):
    files = [c for c in csvs if f"flows-{iv}-sec" in c]
    d = pd.concat([pd.read_csv(c, low_memory=False) for c in files], ignore_index=True)
    ysub = d["SubCategoryLabel"].astype(str).str.strip()
    y9 = ysub.where(~ysub.str.startswith("benign"), "benign")   # fusion des 3 variantes benignes
    t = pd.to_numeric(d["StartTime"], errors="coerce")
    assert t.notna().all()
    return d, y9, t

df, y9_raw, t_start = load_slice("60")
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder(); y = le.fit_transform(y9_raw)
CLASS_NAMES = list(le.classes_); C = len(CLASS_NAMES)
BENIGN_IDX = CLASS_NAMES.index("benign")
t_np = t_start.values.astype(np.float64)

print(f"tranche 60 s : {len(df):,} flux, {df.shape[1]} colonnes brutes, {C} classes")
dist = pd.DataFrame({"flux": pd.Series(y9_raw).value_counts(),
                     "part (%)": (pd.Series(y9_raw).value_counts(normalize=True)*100).round(2)})
display(dist)

fig, ax = plt.subplots(figsize=(7, 3))
dist["flux"].plot.bar(ax=ax, color=["seagreen" if c == "benign" else "indianred" for c in dist.index])
ax.set_yscale("log"); ax.set_ylabel("flux (log)")
ax.set_title("Distribution naturelle des 9 classes (60 s) — desequilibre ~20:1")
plt.tight_layout(); plt.savefig(FIGS / "fig_eda_classes.png"); plt.show()


In [ ]:
# 2.3 — Structure temporelle : chronologie et fenetres d'activite par classe
def activity_windows(times, gap=WINDOW_GAP_S):
    ts = np.sort(times)
    brk = np.where(np.diff(ts) > gap)[0]
    return [(seg[0], seg[-1], len(seg)) for seg in np.split(ts, brk + 1)]

t0 = t_np.min()
win_rows = []
for i, cn in enumerate(CLASS_NAMES):
    w = activity_windows(t_np[y == i])
    win_rows.append({"classe": cn, "fenetres": len(w),
                     "duree totale (h)": round(sum(e - s for s, e, _ in w) / 3600, 2),
                     "premiere (h)": round((w[0][0] - t0) / 3600, 1),
                     "derniere (h)": round((w[-1][1] - t0) / 3600, 1)})
twin = pd.DataFrame(win_rows).set_index("classe")
display(twin)
RESULTS["activity_windows"] = twin.reset_index().to_dict("records")

fig, ax = plt.subplots(figsize=(9, 3.4))
rng = np.random.RandomState(0)
for i, cn in enumerate(CLASS_NAMES):
    ix = np.where(y == i)[0]; ix = rng.choice(ix, size=min(3000, len(ix)), replace=False)
    ax.plot((t_np[ix] - t0) / 3600, np.full(len(ix), i), "|", ms=4,
            color="seagreen" if i == BENIGN_IDX else "indianred", alpha=0.25)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("heures depuis le debut de capture (6-12 fevrier 2025)")
ax.set_title("Figure 1 — fenetres temporelles par classe : etroites et disjointes")
plt.tight_layout(); plt.savefig(FIGS / "fig1_timeline.png"); plt.savefig(FIGS / "fig1_timeline.pdf"); plt.show()
save_results()
# Lecture : chaque famille d'attaque occupe des fenetres etroites. Consequence directe :
# la POSITION temporelle d'un flux suffit presque a predire sa classe (verifie en §3.5),
# et un split chronologique global est degenere (verifie en §3.4).


## 3. Prétraitement

**Catégorisation explicite des colonnes** (listes nominatives, pas de motifs — deux bugs
du pipeline initial sont ainsi corrigés : `StartTime`/`LastTime` passaient comme features,
et le motif `"id"` supprimait par accident `SIntPktIdl`/`DIntPktIdl`) :

| Catégorie | Exemples | Traitement |
|---|---|---|
| **Identifiants** (IP, MAC, ports, n° de flux…) | `SrcAddr`, `Dport`, `FlowID` | exclus de toutes les conditions |
| **Positionnelles** (position dans la capture) | `StartTime`, `LastTime`, `Rank`, `Seq` | condition `full` seulement |
| **Comportementales** (durées, volumes, TTL…) | `Dur`, `TotPkts`, `sTtl` | candidates ; l'audit (§5.2) tranche |
| Labels | `BinaryLabel`, … | jamais features |


In [ ]:
# 3.1 — Jeux de features
IDENT_LIST = ["FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",
              "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",
              "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",
              "sCo", "dCo", "sVid", "dVid"]
POS_LIST   = ["StartTime", "LastTime", "Rank", "Seq"]
LAB_LIST   = ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"]

def feature_sets(d):
    ident = [c for c in IDENT_LIST if c in d.columns]
    pos   = [c for c in POS_LIST if c in d.columns]
    num = (d.drop(columns=ident + LAB_LIST, errors="ignore")
             .select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan))
    const = num.nunique(dropna=True); dropped_const = const[const <= 1].index.tolist()
    num = num.drop(columns=dropped_const).fillna(0.0).astype(np.float32)
    full = list(num.columns); clean = [c for c in full if c not in pos]
    return num, full, clean, pos, ident, dropped_const

X_all, F_FULL, F_CLEAN, POSITIONAL, IDENTIFIERS, DROPPED_CONST = feature_sets(df)
print(f"features : full={len(F_FULL)} | clean={len(F_CLEAN)} "
      f"| positionnelles exclues de clean : {POSITIONAL}")
print(f"identifiants exclus partout ({len(IDENTIFIERS)}), constantes ecartees ({len(DROPPED_CONST)})")

# redondance (information pour la discussion du papier)
sub = X_all[F_CLEAN].sample(min(20000, len(X_all)), random_state=0)
corr = np.corrcoef(sub.values, rowvar=False)
hi = int((np.abs(np.triu(corr, 1)) > 0.95).sum())
print(f"paires de features |r|>0.95 (echantillon 20k) : {hi}")

RESULTS["slice60"] = {"n": int(len(y)), "classes": CLASS_NAMES,
                      "class_counts": pd.Series(y9_raw).value_counts().to_dict(),
                      "benign_share": float((y == BENIGN_IDX).mean()),
                      "features_full": F_FULL, "features_clean": F_CLEAN,
                      "positional": POSITIONAL, "identifiers_excluded": IDENTIFIERS,
                      "high_corr_pairs": hi}
save_results()


In [ ]:
# 3.2 — Splits geles : stratifie x5 + TEMPOREL PAR CLASSE (+ diagnostic chrono global)
from sklearn.model_selection import train_test_split

def temporal_split_per_class(y_, t_, frac=SPLIT_FRAC):
    # pour chaque classe : tri temporel, 60/20/20 -> le test est posterieur au train
    tr, va, te = [], [], []
    for c in range(C):
        ix = np.where(y_ == c)[0]
        ix = ix[np.argsort(t_[ix], kind="stable")]
        n = len(ix); a = int(frac[0] * n); b = int((frac[0] + frac[1]) * n)
        tr.append(ix[:a]); va.append(ix[a:b]); te.append(ix[b:])
    return np.concatenate(tr), np.concatenate(va), np.concatenate(te)

SPLITS_PATH = SAVE / "frozen_splits_60s_v2.npz"
KEYS = [f"strat_seed{s}" for s in SEEDS] + ["temporal"]
if SPLITS_PATH.exists():
    z = np.load(SPLITS_PATH)
    splits = {k: (z[f"{k}_train"], z[f"{k}_val"], z[f"{k}_test"]) for k in KEYS}
    print("splits recharges (geles)")
else:
    old = SAVE / "frozen_splits_60s.npz"          # reutilise les stratifies deja geles
    splits = {}
    if old.exists():
        z = np.load(old)
        for s in SEEDS:
            k = f"strat_seed{s}"
            splits[k] = (z[f"{k}_train"], z[f"{k}_val"], z[f"{k}_test"])
        print("stratifies recuperes du gel precedent")
    else:
        idx = np.arange(len(y))
        for s in SEEDS:
            itr, itmp = train_test_split(idx, test_size=0.40, random_state=s, stratify=y)
            iva, ite = train_test_split(itmp, test_size=0.50, random_state=s, stratify=y[itmp])
            splits[f"strat_seed{s}"] = (itr, iva, ite)
    splits["temporal"] = temporal_split_per_class(y, t_np)
    np.savez_compressed(SPLITS_PATH, **{f"{k}_{p}": v for k, (a, b, c_) in splits.items()
                                        for p, v in zip(("train", "val", "test"), (a, b, c_))})
    print("splits geles ->", SPLITS_PATH)

# verification du split temporel : chaque classe presente partout, test posterieur au train
tr, va, te = splits["temporal"]
chk = pd.DataFrame({"train": pd.Series(y[tr]).value_counts().reindex(range(C), fill_value=0).values,
                    "val":   pd.Series(y[va]).value_counts().reindex(range(C), fill_value=0).values,
                    "test":  pd.Series(y[te]).value_counts().reindex(range(C), fill_value=0).values},
                   index=CLASS_NAMES)
display(chk)
gap_ok = all(t_np[tr][y[tr] == c].max() <= t_np[te][y[te] == c].min() for c in range(C)
             if (y[tr] == c).any() and (y[te] == c).any())
print("invariant temporel (test posterieur au train, par classe) :", "OK" if gap_ok else "VIOLE")
RESULTS["temporal_class_table"] = chk.to_dict()
save_results()


In [ ]:
# 3.3 — Pourquoi pas un split chronologique GLOBAL ? Demonstration (resultat du papier)
o = np.argsort(t_np, kind="stable"); n = len(o)
g_tr, g_te = o[:int(.6*n)], o[int(.8*n):]
gtab = pd.DataFrame({"train (60% premiers)": pd.Series(y[g_tr]).value_counts().reindex(range(C), fill_value=0).values,
                     "test (20% derniers)":  pd.Series(y[g_te]).value_counts().reindex(range(C), fill_value=0).values},
                    index=CLASS_NAMES)
display(gtab)
missing = [CLASS_NAMES[c] for c in range(C) if gtab.iloc[c, 1] > 0 and gtab.iloc[c, 0] == 0]
print(f"classes du test jamais vues a l'entrainement : {missing or 'aucune'}")
RESULTS["global_chrono_table"] = gtab.to_dict()
save_results()
# Lecture : le decoupage global met dans le test des blocs entiers de classes inconnues
# (accuracy ~0 pour tout modele) : il mesure la disjonction des fenetres, pas la
# generalisation. C'est un RESULTAT (Section audit du papier), pas un protocole.
# D'ou le split temporel PAR CLASSE de §3.2.


In [ ]:
# 3.4 — Sondes de raccourci : que predit la seule POSITION temporelle ?
from sklearn.tree import DecisionTreeClassifier

def probe(cols, key):
    tr_, _, te_ = splits[key]
    Xp = df[cols].astype(np.float64).values
    clf = DecisionTreeClassifier(random_state=1).fit(Xp[tr_], y[tr_])
    return float((clf.predict(Xp[te_]) == y[te_]).mean())

if "shortcut_probes" not in RESULTS:
    chance = float(pd.Series(y).value_counts(normalize=True).max())
    pr = {"chance_majority": chance}
    for nm, cols in [("starttime_only", ["StartTime"]), ("positional_all", POSITIONAL)]:
        accs = [probe(cols, f"strat_seed{s}") for s in SEEDS]
        pr[nm] = {"strat_mean": float(np.mean(accs)), "strat_std": float(np.std(accs)),
                  "temporal": probe(cols, "temporal")}
    RESULTS["shortcut_probes"] = pr; save_results()
pr = RESULTS["shortcut_probes"]
print(f"hasard (classe majoritaire)        : {pr['chance_majority']:.3f}")
for nm in ("starttime_only", "positional_all"):
    print(f"{nm:18s} accuracy 9 classes : stratifie {pr[nm]['strat_mean']:.4f} "
          f"(+/- {pr[nm]['strat_std']:.4f}) | temporel {pr[nm]['temporal']:.4f}")
# Lecture : si StartTime seul atteint ~0.99 en stratifie et s'effondre en temporel,
# le split stratifie recompense la MEMORISATION DU CALENDRIER de capture.
# C'est le chiffre d'accroche de l'introduction du papier (RQ2).


## 4. Modèles

Neuf familles supervisées + un détecteur non supervisé, toutes évaluées sous le même protocole :

| Modèle | Famille | Rôle dans le benchmark | Hyperparamètres (déclarés) |
|---|---|---|---|
| `majority` | baseline triviale | plancher de référence | — |
| `logreg` | linéaire | baseline calibrable | `max_iter=1000` |
| `nb` | bayésien naïf | baseline probabiliste classique (couverture Maseer et al. 2021) | GaussianNB |
| `knn` | plus proches voisins | baseline à mémoire (couverture Maseer et al. 2021 ; budget limité, voir §5.1) | k=5 |
| `rf` | forêt aléatoire | référence arbres « sac » | 200 arbres |
| `xgboost` | boosting | état de l'art tabulaire | 300 arbres, prof. 8, lr 0.1 |
| `lightgbm` | boosting | état de l'art tabulaire | 300 arbres, 63 feuilles, lr 0.1 |
| `rnn` | récurrent | **provenance BAg-IDS** (arch. identique) | SimpleRNN(64)+Dense(64) |
| `cnn` | convolutif 1D | **provenance BAg-IDS** | Conv(64,3)→Conv(32,3)→Dense(64) |
| `dnn` | dense | **provenance BAg-IDS** | 128→64→softmax |
| `ftt` | FT-Transformer | représentant profond tabulaire moderne | d=64, 3 blocs, 8 têtes |
| `ae` | autoencodeur | anomalie (bénin seul), évalué à part en AUROC | 64-32-16-32-64 |

Entraînement profond : Adam, 30 époques max, early stopping (patience 5), `class_weight`
équilibré (compense le déséquilibre **à l'entraînement seulement** — le test reste naturel).

**Exclusions déclarées** (à reprendre dans le papier) : le SVM à noyau est exclu
(complexité d'entraînement O(n²), inentraînable sur ~200 000 flux à budget égal ; la
régression logistique couvre la famille linéaire) ; le trio non supervisé k-means/EM/SOM
de Maseer et al. est remplacé par l'autoencodeur, représentant moderne de la détection
d'anomalies, évalué à part en AUROC.


In [ ]:
# 4.1 — Definitions
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                             roc_auc_score, confusion_matrix)

def sk_zoo():
    return {"majority": DummyClassifier(strategy="most_frequent"),
            "logreg":   LogisticRegression(max_iter=1000, n_jobs=-1),
            "nb":       GaussianNB(),
            "knn":      KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
            "rf":       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
            "xgboost":  XGBClassifier(tree_method="hist", n_estimators=300, max_depth=8,
                                      learning_rate=0.1, n_jobs=-1, random_state=0,
                                      eval_metric="mlogloss"),
            "lightgbm": LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                                       n_jobs=-1, random_state=0, verbose=-1)}
FAST = list(sk_zoo().keys())

def build_dnn(F):   # architectures BAg-IDS : NE PAS MODIFIER (provenance)
    return models.Sequential([layers.Input((F,)),
        layers.Dense(128, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"), layers.Dropout(0.2),
        layers.Dense(C, activation="softmax")], name="dnn")
def build_cnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.Conv1D(64, 3, activation="relu", padding="same"), layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, activation="relu", padding="same"), layers.Flatten(),
        layers.Dense(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(C, activation="softmax")], name="cnn")
def build_rnn(F):
    return models.Sequential([layers.Input((F, 1)),
        layers.SimpleRNN(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(C, activation="softmax")], name="rnn")

class FeatureTokenizer(layers.Layer):     # FT-Transformer (Gorishniy et al. 2021)
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        F = int(shape[-1])
        self.w = self.add_weight(shape=(F, self.d), initializer="glorot_uniform", name="w")
        self.b = self.add_weight(shape=(F, self.d), initializer="zeros", name="b")
    def call(self, x): return x[:, :, None] * self.w + self.b
class ClsToken(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer="glorot_uniform", name="cls")
    def call(self, x):
        return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
def build_ftt(F, d=64, heads=8, blocks=3, ff=128, drop=0.1):
    inp = layers.Input((F,))
    x = ClsToken(d)(FeatureTokenizer(d)(inp))
    for _ in range(blocks):
        h = layers.LayerNormalization()(x)
        h = layers.MultiHeadAttention(num_heads=heads, key_dim=d // heads, dropout=drop)(h, h)
        x = layers.Add()([x, h])
        h = layers.LayerNormalization()(x)
        h = layers.Dense(ff, activation="gelu")(h); h = layers.Dropout(drop)(h)
        x = layers.Add()([x, layers.Dense(d)(h)])
    out = layers.Dense(C, activation="softmax")(layers.LayerNormalization()(x[:, 0]))
    return models.Model(inp, out, name="ftt")

DEEP = {"rnn": build_rnn, "cnn": build_cnn, "dnn": build_dnn, "ftt": build_ftt}
def shape_for(name, A): return A if name in ("dnn", "ftt") else A.reshape(-1, A.shape[1], 1)

def build_ae(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(64, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"), layers.Dense(F)], name="ae")
print("modeles definis :", FAST + list(DEEP) + ["ae"])


In [ ]:
# 4.2 — Boucles d'entrainement et d'evaluation (protocole propre)
def make_xy(cols, key):
    tr_, va_, te_ = splits[key]
    X = X_all[cols].values
    sc = RobustScaler().fit(X[tr_])                       # scaler sur TRAIN uniquement
    parts = [np.nan_to_num(sc.transform(X[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
             for i_ in (tr_, va_, te_)]
    return parts, (y[tr_], y[va_], y[te_])

def evaluate(y_true, probs, fit_t, pred_t):
    pred = probs.argmax(1); present = np.unique(y_true)
    is_att = y_true != BENIGN_IDX; pred_att = pred != BENIGN_IDX
    p_att = 1.0 - probs[:, BENIGN_IDX]
    return {"accuracy": float((pred == y_true).mean()),
            "macro_f1": float(f1_score(y_true, pred, labels=present, average="macro", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in zip(range(C),
                f1_score(y_true, pred, labels=range(C), average=None, zero_division=0))},
            "binary": {"detection_f1": float(f1_score(is_att, pred_att, zero_division=0)),
                       "fpr": float(pred_att[~is_att].mean()) if (~is_att).any() else None,
                       "fnr": float((~pred_att[is_att]).mean()) if is_att.any() else None,
                       "pr_auc": float(average_precision_score(is_att, p_att))
                                 if 0 < is_att.mean() < 1 else None},
            "fit_time_s": round(fit_t, 2), "predict_time_s": round(pred_t, 3)}

def done(key): return key in RESULTS["models"] and (PROBS / f"{key.replace('|','_')}.npz").exists()

def record(key, y_true, pva, pte, fit_t, pred_t):
    RESULTS["models"][key] = evaluate(y_true, pte, fit_t, pred_t)
    np.savez_compressed(PROBS / f"{key.replace('|','_')}.npz",
                        probs_val=pva.astype(np.float16), probs_test=pte.astype(np.float16))
    r = RESULTS["models"][key]
    fpr = r["binary"]["fpr"]; fs = f"{fpr:.4%}" if fpr is not None else "n/a"
    print(f"  {key:40s} acc {r['accuracy']:.4f}  mF1 {r['macro_f1']:.4f}  "
          f"MCC {r['mcc']:.4f}  FPR {fs}  [{fit_t:.0f}s]")
    save_results()

def fit_sk(mname, Xtr, ytr):
    # Fit sklearn/xgboost avec remapping des classes vers 0..k-1 (necessaire pour
    # XGBoost quand une classe manque du train), puis realignement des probabilites.
    present = np.unique(ytr)
    model = sk_zoo()[mname]
    y_fit = np.searchsorted(present, ytr) if len(present) < C else ytr
    model.fit(Xtr, y_fit)
    def predict_full(X):
        p = model.predict_proba(X)
        if len(present) == C and list(getattr(model, "classes_", range(C))) == list(range(C)):
            return p
        out = np.zeros((len(X), C), dtype=np.float32)
        cols_idx = present if len(present) < C else np.asarray(model.classes_, int)
        out[:, cols_idx] = p
        return out
    return model, predict_full

def train_fast(mname, cols, key_split, tag):
    key = f"{mname}|{tag}|{key_split}"
    if done(key): return
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, key_split)
    t0 = time.time(); model, pf = fit_sk(mname, Xtr, ytr); fit_t = time.time() - t0
    t0 = time.time(); pte = pf(Xte); pred_t = time.time() - t0
    record(key, yte, pf(Xva), pte, fit_t, pred_t)

def class_weights_safe(ytr):
    present = np.unique(ytr)
    w = compute_class_weight("balanced", classes=present, y=ytr)
    cw = {int(c): 1.0 for c in range(C)}; cw.update({int(c): float(v) for c, v in zip(present, w)})
    return cw

def train_deep(mname, cols, key_split, seed, tag):
    key = f"{mname}|{tag}|{key_split}"
    if done(key): return
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(cols, key_split)
    tf.keras.utils.set_random_seed(seed)
    m = DEEP[mname](Xtr.shape[1])
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4 if mname == "ftt" else 1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    m.fit(shape_for(mname, Xtr), ytr, validation_data=(shape_for(mname, Xva), yva),
          epochs=30, batch_size=512 if mname == "ftt" else 256,
          class_weight=class_weights_safe(ytr), verbose=0,
          callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                             restore_best_weights=True)])
    fit_t = time.time() - t0
    t0 = time.time(); pte = m.predict(shape_for(mname, Xte), batch_size=1024, verbose=0)
    pred_t = time.time() - t0
    pva = m.predict(shape_for(mname, Xva), batch_size=1024, verbose=0)
    if tag == "clean" and key_split == "strat_seed1":
        m.save(SAVE / f"{mname}_clean_seed1.keras")
    record(key, yte, pva, pte, fit_t, pred_t)
print("boucles pretes")


## 5. Entraînement

Trois vagues :

1. **§5.1 — Avant audit** : conditions `full` et `clean`, splits stratifiés (les runs des
   sessions précédentes sont réutilisés) + split temporel en `clean`.
2. **§5.2 — Audit** : importance par permutation, puis **test de transférabilité
   mono-feature** — une feature dont le pouvoir prédictif seul s'effondre entre split
   stratifié et split temporel encode la *position*, pas le *comportement* → liste noire.
3. **§5.3 — Après audit** : condition `audited`, stratifié × 5 + temporel — le tableau
   principal du papier.


In [ ]:
# 5.1 — Avant audit : full + clean
COND = {"full": F_FULL, "clean": F_CLEAN}
STRATS = [f"strat_seed{s}" for s in SEEDS]

print("== modeles rapides ==")
# Budget declare : la prediction k-NN est en O(n_train x n_test) ; avant audit,
# k-NN n'est evalue que sur la graine 1 (la condition auditee, elle, est complete).
for tag, cols in COND.items():
    for sk_ in STRATS + (["temporal"] if tag == "clean" else []):
        for mname in FAST:
            if mname == "knn" and sk_ != "strat_seed1":
                continue
            train_fast(mname, cols, sk_, tag)

print("== modeles profonds ==")
for mname in DEEP:
    for s in SEEDS:
        train_deep(mname, F_CLEAN, f"strat_seed{s}", s, "clean")
    train_deep(mname, F_CLEAN, "temporal", 1, "clean")
    train_deep(mname, F_FULL, "strat_seed1", 1, "full")
print("§5.1 termine")


In [ ]:
# 5.2 — Audit : permutation + transferabilite mono-feature -> liste noire
from sklearn.inspection import permutation_importance

if "audit" not in RESULTS:
    (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_CLEAN, "strat_seed1")
    ref = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                         n_jobs=-1, random_state=0, verbose=-1).fit(Xtr, ytr)
    rng = np.random.RandomState(0)
    sub_ix = rng.choice(len(yte), size=min(20000, len(yte)), replace=False)
    imp = permutation_importance(ref, Xte[sub_ix], yte[sub_ix], n_repeats=5,
                                 random_state=0, n_jobs=-1, scoring="accuracy")
    order = np.argsort(-imp.importances_mean)
    top = [(F_CLEAN[i], float(imp.importances_mean[i])) for i in order[:20]]

    chance = RESULTS["shortcut_probes"]["chance_majority"]
    rows, flagged = [], []
    for feat, im in top:
        a_s = probe([feat], "strat_seed1"); a_t = probe([feat], "temporal")
        shortcut = (a_s > 3 * chance) and (a_t < 0.5 * a_s)
        rows.append({"feature": feat, "perm_imp": round(im, 4),
                     "acc seule (stratifie)": round(a_s, 3),
                     "acc seule (temporel)": round(a_t, 3),
                     "raccourci": "OUI" if shortcut else "-"})
        if shortcut: flagged.append(feat)
    audit_tab = pd.DataFrame(rows)
    display(audit_tab)
    RESULTS["audit"] = {"version": "v2-temporal", "perm_importance_top": top,
                        "transfer_table": rows, "blacklist": POSITIONAL + flagged}
    save_results()
BLACKLIST = RESULTS["audit"]["blacklist"]
F_AUDIT = [c for c in F_CLEAN if c not in BLACKLIST]
RESULTS["slice60"]["features_audited"] = F_AUDIT; save_results()
print(f"\nLISTE NOIRE ({len(BLACKLIST)}) : {BLACKLIST}")
print(f"features auditees restantes : {len(F_AUDIT)}")
# Lecture : la liste noire = colonnes positionnelles + toute feature comportementale
# dont le pouvoir predictif ne survit pas au passage stratifie -> temporel.
# C'est l'artefact publiable central du papier (RQ2).


In [ ]:
# 5.3 — Apres audit : condition auditee (tableau principal du papier)
print("== modeles rapides ==")
for sk_ in STRATS + ["temporal"]:
    for mname in FAST:
        train_fast(mname, F_AUDIT, sk_, "audited")
print("== modeles profonds ==")
for mname in DEEP:
    for s in SEEDS:
        train_deep(mname, F_AUDIT, f"strat_seed{s}", s, "audited")
    train_deep(mname, F_AUDIT, "temporal", 1, "audited")
print("§5.3 termine")


In [ ]:
# 5.4 — Autoencodeur (benin seul) : AUROC global et par famille
if "autoencoder" not in RESULTS:
    ae_res = {}
    for s in SEEDS:
        (Xtr, Xva, Xte), (ytr, yva, yte) = make_xy(F_AUDIT, f"strat_seed{s}")
        tf.keras.utils.set_random_seed(s)
        ae = build_ae(Xtr.shape[1]); ae.compile(optimizer="adam", loss="mse")
        ae.fit(Xtr[ytr == BENIGN_IDX], Xtr[ytr == BENIGN_IDX],
               validation_data=(Xva[yva == BENIGN_IDX], Xva[yva == BENIGN_IDX]),
               epochs=50, batch_size=256, verbose=0,
               callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                  restore_best_weights=True)])
        err = np.mean((ae.predict(Xte, batch_size=1024, verbose=0) - Xte) ** 2, axis=1)
        per_fam = {CLASS_NAMES[i]: float(roc_auc_score(
                       (yte[(yte == i) | (yte == BENIGN_IDX)] == i),
                       err[(yte == i) | (yte == BENIGN_IDX)]))
                   for i in range(C) if i != BENIGN_IDX and (yte == i).any()}
        ae_res[f"seed{s}"] = {"auroc_global": float(roc_auc_score(yte != BENIGN_IDX, err)),
                              "auroc_per_family": per_fam}
        print(f"  AE seed {s} : AUROC {ae_res[f'seed{s}']['auroc_global']:.4f}")
    RESULTS["autoencoder"] = ae_res; save_results()
g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"AE AUROC global : {np.mean(g):.4f} +/- {np.std(g):.4f}")


## 6. Évaluation, figures et interprétation

Toutes les figures sont exportées en PNG **et** PDF à 300 dpi dans `figures/`,
les fragments de tables LaTeX dans `tables/` — prêts pour `paper/main.tex`.


In [ ]:
# 6.1 — Tableau principal : macro-F1 (moyenne +/- ecart-type, 5 graines) par condition
ALL_MODELS = FAST + list(DEEP)

def agg(mname, tag, metric="macro_f1"):
    vals = [RESULTS["models"][f"{mname}|{tag}|strat_seed{s}"][metric]
            for s in SEEDS if f"{mname}|{tag}|strat_seed{s}" in RESULTS["models"]]
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (np.nan, np.nan)

def one(mname, tag, split, metric="macro_f1"):
    return RESULTS["models"].get(f"{mname}|{tag}|{split}", {}).get(metric, np.nan)

rows = []
for m_ in ALL_MODELS:
    mu_c, sd_c = agg(m_, "clean"); mu_a, sd_a = agg(m_, "audited")
    rows.append({"modele": m_,
                 "full (strat, s1)": round(one(m_, "full", "strat_seed1"), 4),
                 "clean (strat, 5g)": f"{mu_c:.4f} +/- {sd_c:.4f}",
                 "audited (strat, 5g)": f"{mu_a:.4f} +/- {sd_a:.4f}",
                 "clean (temporel)": round(one(m_, "clean", "temporal"), 4),
                 "audited (temporel)": round(one(m_, "audited", "temporal"), 4),
                 "MCC audited s1": round(one(m_, "audited", "strat_seed1", "mcc"), 4)})
main_tab = pd.DataFrame(rows).set_index("modele")
display(main_tab)

# fragments LaTeX
def fmt(x): return "--" if (isinstance(x, float) and np.isnan(x)) else (f"{x:.4f}" if isinstance(x, float) else str(x))
pm = lambda s: str(s).replace("+/-", "$\\pm$")
(TABS / "t2_main.tex").write_text("\n".join(
    f"{i} & {fmt(r['full (strat, s1)'])} & {pm(r['clean (strat, 5g)'])} & "
    f"{pm(r['audited (strat, 5g)'])} & {fmt(r['clean (temporel)'])} & "
    f"{fmt(r['audited (temporel)'])} \\\\"
    for i, r in main_tab.iterrows()))
(TABS / "t1_intervals.tex").write_text("\n".join(
    f"{iv}\\,s & {RESULTS['interval_stats'][iv]['total']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign_share']*100:.1f}\\% \\\\" for iv in INTERVALS))
print("tables LaTeX ecrites :", sorted(p.name for p in TABS.glob('*.tex')))


In [ ]:
# 6.2 — Figure phare : classement avant / apres audit
conds = [("full\n(strat)", lambda m: one(m, "full", "strat_seed1") if m in DEEP else agg(m, "full")[0]),
         ("clean\n(strat)", lambda m: agg(m, "clean")[0]),
         ("audited\n(strat)", lambda m: agg(m, "audited")[0]),
         ("clean\n(temporel)", lambda m: one(m, "clean", "temporal")),
         ("audited\n(temporel)", lambda m: one(m, "audited", "temporal"))]
fig, ax = plt.subplots(figsize=(8, 4.5))
for m_ in ALL_MODELS:
    if m_ == "majority": continue
    ys = [f(m_) for _, f in conds]
    ax.plot(range(len(conds)), ys, marker="o", ms=4, label=m_)
ax.set_xticks(range(len(conds))); ax.set_xticklabels([c for c, _ in conds])
ax.set_ylabel("macro-F1"); ax.set_ylim(bottom=0)
ax.set_title("Figure 2 — l'effet de l'audit et du split temporel sur le classement (60 s)")
ax.legend(fontsize=7, ncol=2, loc="lower left")
plt.tight_layout(); plt.savefig(FIGS / "fig2_before_after.png"); plt.savefig(FIGS / "fig2_before_after.pdf"); plt.show()
# Lecture : l'ecart entre les colonnes stratifiees et temporelles est la part de
# performance attribuable a la memorisation de la capture (reponse a RQ2).


In [ ]:
# 6.3 — Calibration : temperature scaling + ECE + diagrammes de fiabilite
from scipy.optimize import minimize_scalar

def load_probs(key):
    z = np.load(PROBS / f"{key.replace('|','_')}.npz")
    return z["probs_val"].astype(np.float64), z["probs_test"].astype(np.float64)
def fit_temperature(pva, yva):
    logp = np.log(np.clip(pva, 1e-12, 1))
    def nll(T):
        q = logp / T; q -= q.max(1, keepdims=True)
        p = np.exp(q); p /= p.sum(1, keepdims=True)
        return -np.mean(np.log(np.clip(p[np.arange(len(yva)), yva], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x)
def apply_T(p, T):
    q = np.log(np.clip(p, 1e-12, 1)) / T; q -= q.max(1, keepdims=True)
    e = np.exp(q); return e / e.sum(1, keepdims=True)
def ece(p, y_true, bins=15):
    conf = p.max(1); acc = (p.argmax(1) == y_true).astype(float)
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m_ = (conf > lo) & (conf <= hi)
        if m_.any(): out += m_.mean() * abs(acc[m_].mean() - conf[m_].mean())
    return float(out)

MAIN = ["logreg", "nb", "knn", "rf", "xgboost", "lightgbm", "rnn", "cnn", "dnn", "ftt"]
yva_ref, yte_ref = y[splits["strat_seed1"][1]], y[splits["strat_seed1"][2]]
if "calibration" not in RESULTS:
    cal = {}
    for m_ in MAIN:
        key = f"{m_}|audited|strat_seed1"
        if not done(key): continue
        pva, pte = load_probs(key); T = fit_temperature(pva, yva_ref)
        cal[m_] = {"T": T, "ece_before": ece(pte, yte_ref),
                   "ece_after": ece(apply_T(pte, T), yte_ref)}
    RESULTS["calibration"] = cal; save_results()
display(pd.DataFrame(RESULTS["calibration"]).T.round(4))

fig, axes = plt.subplots(2, 5, figsize=(13, 5), sharex=True, sharey=True)
for ax, m_ in zip(axes.flat, MAIN):
    key = f"{m_}|audited|strat_seed1"
    if not done(key) or m_ not in RESULTS["calibration"]:
        ax.axis("off"); continue
    _, pte = load_probs(key); T = RESULTS["calibration"][m_]["T"]
    for p, lab in [(pte, "brut"), (apply_T(pte, T), "calibre")]:
        conf, pred = p.max(1), p.argmax(1)
        xs, ys = [], []
        for lo, hi in zip(np.linspace(0, 1, 16)[:-1], np.linspace(0, 1, 16)[1:]):
            m2 = (conf > lo) & (conf <= hi)
            if m2.any(): xs.append(conf[m2].mean()); ys.append((pred[m2] == yte_ref[m2]).mean())
        ax.plot(xs, ys, marker="o", ms=2.5, label=lab)
    ax.plot([0, 1], [0, 1], "k--", lw=.6); ax.set_title(m_, fontsize=8)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Figure 4 — diagrammes de fiabilite (condition auditee)")
plt.tight_layout(); plt.savefig(FIGS / "fig4_reliability.png"); plt.savefig(FIGS / "fig4_reliability.pdf"); plt.show()


In [ ]:
# 6.4 — Multi-intervalles 5/10/30 s (LightGBM, XGBoost, DNN — condition auditee)
if RUN_INTERVALS:
    RESULTS.setdefault("intervals", {})
    for iv in ["5", "10", "30"]:
        if iv in RESULTS["intervals"]: continue
        print(f"== intervalle {iv} s ==")
        d_iv, y9_iv, t_iv = load_slice(iv)
        Xiv, _, cleanc, _, _, _ = feature_sets(d_iv)
        audc = [c for c in cleanc if c not in BLACKLIST]
        y_iv = le.transform(y9_iv); idx = np.arange(len(y_iv))
        res_iv = {"n": int(len(y_iv)), "benign_share": float((y_iv == BENIGN_IDX).mean()), "runs": {}}
        stp = d_iv["StartTime"].astype(np.float64).values.reshape(-1, 1)
        for s in [1, 2, 3]:
            itr, itmp = train_test_split(idx, test_size=0.4, random_state=s, stratify=y_iv)
            iva_, ite_ = train_test_split(itmp, test_size=0.5, random_state=s, stratify=y_iv[itmp])
            if s == 1:
                dtc = DecisionTreeClassifier(random_state=1).fit(stp[itr], y_iv[itr])
                res_iv["probe_starttime"] = float((dtc.predict(stp[ite_]) == y_iv[ite_]).mean())
            Xa = Xiv[audc].values; sc = RobustScaler().fit(Xa[itr])
            Xtr, Xva_, Xte = [np.nan_to_num(sc.transform(Xa[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
                              for i_ in (itr, iva_, ite_)]
            ytr_, yte_ = y_iv[itr], y_iv[ite_]
            for mname in ["lightgbm", "xgboost"]:
                t0 = time.time(); mdl, pf = fit_sk(mname, Xtr, ytr_); ft = time.time() - t0
                t0 = time.time(); pte = pf(Xte); pt = time.time() - t0
                res_iv["runs"][f"{mname}|seed{s}"] = evaluate(yte_, pte, ft, pt)
            tf.keras.utils.set_random_seed(s)
            m = build_dnn(Xtr.shape[1])
            m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
            t0 = time.time()
            m.fit(Xtr, ytr_, validation_data=(Xva_, y_iv[iva_]), epochs=30, batch_size=256,
                  class_weight=class_weights_safe(ytr_), verbose=0,
                  callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                     restore_best_weights=True)])
            ft = time.time() - t0
            t0 = time.time(); pte = m.predict(Xte, batch_size=1024, verbose=0); pt = time.time() - t0
            res_iv["runs"][f"dnn|seed{s}"] = evaluate(yte_, pte, ft, pt)
            print(f"  seed {s} ok")
        RESULTS["intervals"][iv] = res_iv; save_results()
        del d_iv, Xiv

    det = pd.DataFrame(index=CLASS_NAMES, columns=INTERVALS, dtype=float)
    for iv in INTERVALS:
        if iv == "60":
            pcf = RESULTS["models"].get("lightgbm|audited|strat_seed1", {}).get("per_class_f1", {})
        else:
            pcs = [r["per_class_f1"] for k, r in RESULTS["intervals"].get(iv, {}).get("runs", {}).items()
                   if k.startswith("lightgbm")]
            pcf = {cn: float(np.mean([p[cn] for p in pcs])) for cn in CLASS_NAMES} if pcs else {}
        for cn in CLASS_NAMES: det.loc[cn, iv] = pcf.get(cn, np.nan)
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    im = ax.imshow(det.values.astype(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{iv}s" for iv in INTERVALS])
    ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(C):
        for j in range(4):
            v = det.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v < .6 else "black", fontsize=7)
    plt.colorbar(im, label="F1 par classe (LightGBM, audite)")
    ax.set_title("Figure 3 — detectabilite par famille vs intervalle")
    plt.tight_layout(); plt.savefig(FIGS / "fig3_interval_heatmap.png")
    plt.savefig(FIGS / "fig3_interval_heatmap.pdf"); plt.show()


In [ ]:
# 6.5 — Banc de cout CPU (latence batch-1, debit batch-512, tailles)
if RUN_COST and "cost" not in RESULTS:
    import pickle, io
    cost = {}
    (Xtr, _, Xte), (ytr, _, _) = make_xy(F_AUDIT, "strat_seed1")
    n1, nb = 200, 20
    x1, xb = Xte[:n1], Xte[:512]
    for mname in FAST:
        mdl, pf = fit_sk(mname, Xtr, ytr)
        lat = []
        for i in range(n1):
            t0 = time.perf_counter(); pf(x1[i:i+1]); lat.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        for _ in range(nb): pf(xb)
        buf = io.BytesIO(); pickle.dump(mdl, buf)
        cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                       "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                       "throughput_512": float(nb * 512 / (time.perf_counter() - t0)),
                       "size_mb": buf.getbuffer().nbytes / 1e6}
    (Xtr_c, _, Xte_c), _ = make_xy(F_CLEAN, "strat_seed1")     # les .keras sont en CLEAN
    x1c, xbc = Xte_c[:n1], Xte_c[:512]
    with tf.device("/CPU:0"):
        for mname in DEEP:
            path = SAVE / f"{mname}_clean_seed1.keras"
            if not path.exists(): continue
            m = tf.keras.models.load_model(path, compile=False)
            xin = (lambda a: a) if mname in ("dnn", "ftt") else (lambda a: a.reshape(-1, a.shape[1], 1))
            m.predict(xin(x1c[:8]), verbose=0)
            lat = []
            for i in range(n1):
                t0 = time.perf_counter(); m.predict(xin(x1c[i:i+1]), verbose=0)
                lat.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            for _ in range(nb): m.predict(xin(xbc), verbose=0)
            cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                           "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                           "throughput_512": float(nb * 512 / (time.perf_counter() - t0)),
                           "size_mb": path.stat().st_size / 1e6, "params": int(m.count_params())}
    RESULTS["cost"] = cost; save_results()
if "cost" in RESULTS:
    display(pd.DataFrame(RESULTS["cost"]).T.round(3))
    (TABS / "t4_cost.tex").write_text("\n".join(
        f"{m_} & {c_['lat_p50_ms']:.2f} & {c_['lat_p99_ms']:.2f} & "
        f"{c_['throughput_512']:,.0f} & {c_['size_mb']:.2f} \\\\"
        for m_, c_ in RESULTS["cost"].items()))
    fig, ax = plt.subplots(figsize=(5.5, 4))
    for m_, c_ in RESULTS["cost"].items():
        mf1 = agg(m_, "audited")[0]
        if np.isnan(mf1): continue
        ax.scatter(c_["throughput_512"], mf1, s=25)
        ax.annotate(m_, (c_["throughput_512"], mf1), fontsize=7, xytext=(4, 3),
                    textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("debit CPU (flux/s, batch 512)")
    ax.set_ylabel("macro-F1 (audite)"); ax.set_title("Figure 5 — cout vs performance")
    plt.tight_layout(); plt.savefig(FIGS / "fig5_cost.png"); plt.savefig(FIGS / "fig5_cost.pdf"); plt.show()


In [ ]:
# 6.6 — Tests statistiques + matrice de confusion du meilleur modele
from scipy.stats import chi2 as chi2dist

def mcnemar_p(pa, pb, yt):
    ca, cb = pa == yt, pb == yt
    b_, c_ = int((ca & ~cb).sum()), int((~ca & cb).sum())
    if b_ + c_ == 0: return 1.0
    return float(chi2dist.sf((abs(b_ - c_) - 1) ** 2 / (b_ + c_), 1))

if "stats" not in RESULTS:
    preds = {m_: load_probs(f"{m_}|audited|strat_seed1")[1].argmax(1)
             for m_ in MAIN if done(f"{m_}|audited|strat_seed1")}
    raw = {f"{a}|{b}": mcnemar_p(preds[a], preds[b], yte_ref)
           for a, b in itertools.combinations(sorted(preds), 2)}
    order = sorted(raw, key=raw.get); mtot = len(order)
    holm = {k: min(1.0, raw[k] * (mtot - i)) for i, k in enumerate(order)}
    rng = np.random.RandomState(0); boot = {}
    for m_, pr_ in preds.items():
        vals = [f1_score(yte_ref[ix], pr_[ix], average="macro", zero_division=0)
                for ix in (rng.randint(0, len(yte_ref), len(yte_ref)) for _ in range(1000))]
        boot[m_] = {"macro_f1_ci95": [float(np.percentile(vals, 2.5)),
                                      float(np.percentile(vals, 97.5))]}
    RESULTS["stats"] = {"mcnemar_raw": raw, "mcnemar_holm": holm, "bootstrap": boot}
    save_results()
sig = {k: v for k, v in RESULTS["stats"]["mcnemar_holm"].items() if v < 0.05}
print(f"paires significativement differentes (Holm<0.05) : {len(sig)}/{len(RESULTS['stats']['mcnemar_holm'])}")

best = max((m_ for m_ in ALL_MODELS if m_ != "majority"), key=lambda m_: agg(m_, "audited")[0])
pte = load_probs(f"{best}|audited|strat_seed1")[1]
cm = confusion_matrix(yte_ref, pte.argmax(1), labels=range(C), normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
for i in range(C):
    for j in range(C):
        if cm[i, j] >= 0.01:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm[i, j] > .5 else "black")
ax.set_title(f"Figure 6 — matrice de confusion, {best} (audite, graine 1)")
plt.colorbar(im); plt.tight_layout()
plt.savefig(FIGS / "fig6_confusion.png"); plt.savefig(FIGS / "fig6_confusion.pdf"); plt.show()


## 7. Export et synthèse


In [ ]:
# 7.1 — Chiffres cles pour la redaction + archive a renvoyer
pr = RESULTS["shortcut_probes"]
print("=" * 68)
print("CHIFFRES CLES DE L'ARTICLE")
print("=" * 68)
print(f"corpus 60 s : {RESULTS['slice60']['n']:,} flux, {C} classes, "
      f"benin {RESULTS['slice60']['benign_share']:.1%}")
print(f"sonde StartTime seul   : stratifie {pr['starttime_only']['strat_mean']:.4f} "
      f"| temporel {pr['starttime_only']['temporal']:.4f} | hasard {pr['chance_majority']:.3f}")
print(f"liste noire ({len(BLACKLIST)}) : {BLACKLIST}")
best_mu, best_sd = agg(best, "audited")
print(f"meilleur modele (audite, stratifie) : {best} macro-F1 {best_mu:.4f} +/- {best_sd:.4f}")
print(f"meilleur modele (audite, temporel)  : macro-F1 {one(best, 'audited', 'temporal'):.4f}")
g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"autoencodeur AUROC : {np.mean(g):.4f} +/- {np.std(g):.4f}")
print("=" * 68)

save_results()
tmp = pathlib.Path("/content/export"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
shutil.copytree(FIGS, tmp / "figures"); shutil.copytree(TABS, tmp / "tables")
shutil.copy(RES_PATH, tmp / "article1_results.json")
shutil.make_archive("/content/article1_figures_tables", "zip", tmp)
from google.colab import files
files.download("/content/article1_figures_tables.zip")
files.download(str(RES_PATH))
print("Termine — renvoyer article1_results.json + article1_figures_tables.zip")


---
## À renvoyer

1. **`article1_results.json`** — tous les chiffres du papier.
2. **`article1_figures_tables.zip`** — figures 300 dpi (PNG + PDF) et fragments LaTeX.
3. Toute cellule en erreur, avec son message.

## Guide de lecture des résultats

- **§3.4** : si la sonde `StartTime` fait ~0,99 en stratifié et s'effondre en temporel → le chiffre d'accroche de l'introduction est confirmé.
- **§5.2** : la table de transférabilité liste les features dont le pouvoir prédictif ne survit pas au split temporel — chaque ligne « OUI » est une entrée de la liste noire publiée.
- **§6.2 (Figure 2)** : l'écart entre colonnes stratifiées et temporelles = la part de performance due à la mémorisation de la capture. C'est la réponse quantitative à RQ2.
- **Saturation attendue** : en stratifié, les arbres resteront ≥ 0,999 même après audit si les classes sont volumétriquement séparables — c'est un résultat en soi (le corpus est facile *en stratifié*), et c'est le split temporel qui départage réellement les modèles.

En cas de déconnexion : relancer *Tout exécuter* — reprise automatique.
